# Fetch CPT Data from NGL Database Using DAPI

This notebook fetches Cone Penetration Test (CPT) data from the Next Generation Liquefaction (NGL) database for clustering analysis.

> **This notebook needs a DesignSafe account.** It reads `DESIGNSAFE_USERNAME` and
> `DESIGNSAFE_PASSWORD` from a `.env` file and opens a MySQL connection to the NGL
> host, so it does not run in Colab and it does not run offline. The data it produces,
> `all_cpt_profiles.pkl` and `cpt_summary.csv`, is committed next to notebook 06b, so
> you can read this notebook for reference and go straight to 06b.

## Based on:
- **Paper**: Hudson et al. (2023) - "Unsupervised machine learning for detecting soil layer boundaries from cone penetration test data." *Earthquake Engineering & Structural Dynamics*, 52:3201-3215.
- **Data Source**: NGL Database via DesignSafe DAPI

## Objectives:
1. Connect to NGL database using `dapi`
2. Fetch CPT profiles from the sites with the most CPT soundings
3. Calculate derived parameters (qt, Qtn, Fr, Ic, qc1Ncs)
4. Save data locally for offline clustering analysis

## Which CPT profiles?
The paper analyzed **272 CPT profiles** from the NGL database. Cell 13 below takes the
three sites with the most CPT tests, whatever they turn out to be, rather than a list
of named sites. On the run that produced the committed data those three were **Wildlife
Array** (24 soundings), **Moss Landing Marine Lab** (22) and **Jefferson Ranch** (18),
for 64 profiles and 11,191 measurement points in total.


## 1. Install DAPI Package

In [ ]:
%pip install setuptools
%pip install "dapi==0.4.9"


### ⚠️ Restart Kernel After Installation

Please **restart the kernel** before continuing:
- Jupyter Lab: `Kernel` → `Restart Kernel`
- Jupyter Notebook: `Kernel` → `Restart`

Run this notebook in a fresh environment, not in the environment that builds the course
site. `dapi` pulls in `mkdocs`, `mkdocstrings` and `griffe` as hard dependencies, and
installing it alongside the site build can change those versions underneath it.


## 2. Import Libraries

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dapi import DSClient
from dapi import FileOperationError, JobSubmissionError, JobMonitorError
import pickle
from datetime import datetime
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 3. Load Credentials and Connect to NGL Database

### First, set up your `.env` file:

Create a file named `.env` in the root directory with:
```
DESIGNSAFE_USERNAME=your_username
DESIGNSAFE_PASSWORD=your_password
```

**Note**: The `.env` file is gitignored for security.

In [ ]:
# Load credentials from .env file
env_path = '.env'  # Adjust path if needed

if os.path.exists(env_path):
    load_dotenv(env_path)
    print("Credentials loaded from .env file")
else:
    print("No .env file found. Please create one with your DesignSafe credentials.")
    print("  See .env.template for the format.")

# Set environment variables for dapi
# Dapi uses these automatically
username = os.getenv('DESIGNSAFE_USERNAME')
password = os.getenv('DESIGNSAFE_PASSWORD')

# Print only whether the credentials are there. Never print the username or any part
# of the password: this notebook is committed and rendered on the course site.
print(f"Credentials found: {bool(username and password)}")


In [ ]:
# Initialize DSClient and connect to databases
try:
    print("Initializing DSClient...")
    ds = DSClient()
    print("✓ DSClient initialized successfully")
    
    # Create convenience variables for databases
    print("\nConnecting to databases...")
    ngl = ds.db.ngl
    print("✓ Connected to NGL database")
    
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise SystemExit("Stopping notebook due to connection failure")

## 4. Explore Available Sites with CPT Data

Let's identify sites with good CPT coverage, especially those mentioned in the paper.

In [ ]:
# Query sites with CPT data
sql_sites = """
SELECT DISTINCT
    SITE.SITE_ID,
    SITE.SITE_NAME,
    SITE.SITE_LAT,
    SITE.SITE_LON,
    SITE.SITE_GEOL,
    COUNT(DISTINCT TEST.TEST_ID) as num_cpt_tests
FROM SITE
INNER JOIN TEST ON SITE.SITE_ID = TEST.SITE_ID
INNER JOIN SCPG ON SCPG.TEST_ID = TEST.TEST_ID
GROUP BY SITE.SITE_ID, SITE.SITE_NAME, SITE.SITE_LAT, SITE.SITE_LON, SITE.SITE_GEOL
HAVING num_cpt_tests >= 5
ORDER BY num_cpt_tests DESC
LIMIT 30
"""

try:
    sites_df = ngl.read_sql(sql_sites, output_type="DataFrame")
except Exception as e:
    raise SystemExit(
        "The NGL query failed, so every cell below has nothing to work on.\n"
        f"  {type(e).__name__}: {e}\n"
        "The NGL database is reachable from DesignSafe with a valid account. Check that\n"
        "  1. .env holds DESIGNSAFE_USERNAME and DESIGNSAFE_PASSWORD,\n"
        "  2. you are on a network that can reach the NGL host,\n"
        "  3. your DesignSafe account has NGL database access.\n"
        "To skip this notebook, open 06b-clustering-analysis.ipynb: it reads the\n"
        "committed all_cpt_profiles.pkl and cpt_summary.csv."
    )

print(f"\nSites with at least 5 CPT tests (Top 30):")
print("="*100)
sites_df


In [ ]:
# Search for specific sites mentioned in the paper
search_terms = ['Moss Landing', 'Sandholdt', 'Edgecumbe', 'Urayasu', 'Inage']

print("\nSearching for sites mentioned in Hudson et al. (2023):")
print("="*100)

for term in search_terms:
    # Use double %% to escape the SQL LIKE wildcard
    sql = f"""
    SELECT DISTINCT 
        SITE.SITE_ID, 
        SITE.SITE_NAME,
        COUNT(DISTINCT TEST.TEST_ID) as num_tests
    FROM SITE 
    INNER JOIN TEST ON SITE.SITE_ID = TEST.SITE_ID 
    INNER JOIN SCPG ON SCPG.TEST_ID = TEST.TEST_ID
    WHERE SITE.SITE_NAME LIKE '%%{term}%%'
    GROUP BY SITE.SITE_ID, SITE.SITE_NAME
    """
    result = ngl.read_sql(sql, output_type="DataFrame")
    if len(result) > 0:
        print(f"\n'{term}':")
        print(result.to_string(index=False))

## 5. Select Sites for Analysis

Based on the paper and availability, we'll select CPT profiles from sites with:
1. **Documented liquefaction manifestation** (important for context)
2. **Multiple CPT soundings** (to show consistency)
3. **Varying depths** (to test depth-independence of the algorithm)
4. **Different soil types** (to test versatility)

**Target**: 20-30 CPT profiles from 2-3 sites

In [ ]:
# Select specific sites (adjust SITE_IDs based on your query results)
# Example: We'll select top 3 sites with most CPTs
selected_site_ids = sites_df['SITE_ID'].head(3).tolist()

print(f"Selected sites: {selected_site_ids}")
print("\nSite details:")
for site_id in selected_site_ids:
    site_info = sites_df[sites_df['SITE_ID'] == site_id].iloc[0]
    print(f"  SITE_ID {site_id}: {site_info['SITE_NAME']} ({site_info['num_cpt_tests']} CPTs)")

In [ ]:
# Get all CPT tests from selected sites
site_ids_str = ','.join(map(str, selected_site_ids))

sql_tests = f"""
SELECT DISTINCT 
    SITE.SITE_ID,
    SITE.SITE_NAME,
    TEST.TEST_ID,
    TEST.TEST_NAME,
    SCPG.SCPG_ID,
    SCPG.SCPG_CSA,
    SCPG.SCPG_RATE
FROM SITE
INNER JOIN TEST ON SITE.SITE_ID = TEST.SITE_ID
INNER JOIN SCPG ON TEST.TEST_ID = SCPG.TEST_ID
WHERE SITE.SITE_ID IN ({site_ids_str})
ORDER BY SITE.SITE_ID, TEST.TEST_NAME
"""

tests_df = ngl.read_sql(sql_tests, output_type="DataFrame")
print(f"\nFound {len(tests_df)} CPT tests:")
print("="*100)
tests_df

## 6. Functions to Fetch and Process CPT Data

Two things to note in the code below.

Points where a correlation is evaluated outside the range it was fitted on are set to
NaN, not clipped to the bound. Robertson's chart runs from Qtn = 1 to 1000 and from
Fr = 0.1% to 10%; a value pinned to one of those edges is a property of the bound, not
of the soil, and Ic computed from it means nothing. Notebook 06b drops those points
before clustering.

`unit_weight`, `gwt_depth` and `fc` are the same fixed numbers for every profile at
every site. They set the total and effective vertical stress, so they propagate into
Qtn, Ic and qc1Ncs. Holding the fines content at 15% while computing Ic is
self-contradictory, since Ic is the usual way to estimate the fines content in the
first place.


In [ ]:
def fetch_cpt_measurements(scpg_id, ngl_db):
    """
    Fetch CPT measurement data for a given SCPG_ID.

    Parameters:
    -----------
    scpg_id : int
        SCPG ID from NGL database
    ngl_db : DSDatabase
        NGL database connection

    Returns:
    --------
    DataFrame with columns: depth, qc, fs, u2
    """
    sql = f"""
    SELECT
        SCPT.SCPT_DPTH as depth,
        SCPT.SCPT_RES as qc,
        SCPT.SCPT_FRES as fs,
        SCPT.SCPT_PWP as u2
    FROM SCPT
    WHERE SCPT.SCPG_ID = {scpg_id}
    ORDER BY SCPT.SCPT_DPTH
    """

    df = ngl_db.read_sql(sql, output_type="DataFrame")

    # u2 is missing for soundings taken with a cone that has no pore pressure
    # element. Zero is the right substitute there: qt = qc + u2 (1 - a) reduces
    # to qt = qc, which is what an ordinary cone measures.
    df['u2'] = df['u2'].fillna(0)

    # fs is not optional. Fr = fs / (qt - sigma_v) is undefined without it, and
    # filling with zero drives Fr onto its lower bound and makes Ic an artifact
    # of that bound. Drop those points instead.
    n_before = len(df)
    df = df.dropna(subset=['fs']).reset_index(drop=True)
    if len(df) < n_before:
        print(f"  dropped {n_before - len(df)} of {n_before} points with no fs")

    return df


def calculate_derived_parameters(df, unit_weight=18.0, gwt_depth=2.0, a_ratio=0.7, fc=15.0):
    """
    Calculate derived CPT parameters following Hudson et al. (2023).

    unit_weight, gwt_depth and fc are held at the same value for every profile at
    every site. They are assumptions, not measurements. They set sigma_v and
    sigma_v_eff, which propagate into Qtn, Ic and qc1Ncs, so a wrong water table
    or unit weight shifts every derived number in this notebook. A per-site water
    table, and a fines content estimated from Ic rather than fixed at 15%, would
    both be better.

    Points where a correlation is evaluated outside the range it was fitted on are
    set to NaN, not clipped to the bound. A clipped value looks like a measurement
    and is not one.

    Parameters:
    -----------
    df : DataFrame
        CPT measurements with columns: depth, qc, fs, u2
    unit_weight : float
        Total unit weight of soil (kN/m3)
    gwt_depth : float
        Groundwater table depth (m)
    a_ratio : float
        Net area ratio for cone tip
    fc : float
        Fines content (%) for qc1Ncs calculation

    Returns:
    --------
    DataFrame with added derived parameters
    """
    data = df.copy()

    # Convert from MPa to kPa
    data['qc_kpa'] = data['qc'] * 1000.0
    data['fs_kpa'] = data['fs'] * 1000.0
    data['u2_kpa'] = data['u2'] * 1000.0

    # Calculate stresses
    data['sigma_v'] = data['depth'] * unit_weight  # Total vertical stress (kPa)
    data['u0'] = np.where(data['depth'] > gwt_depth,
                         (data['depth'] - gwt_depth) * 9.81,
                         0.0)  # Hydrostatic pore pressure
    data['sigma_v_eff'] = data['sigma_v'] - data['u0']  # Effective stress

    # Corrected tip resistance (Eq. 1)
    data['qt'] = data['qc_kpa'] + data['u2_kpa'] * (1 - a_ratio)

    pa = 101.325  # Atmospheric pressure (kPa)

    # Net tip resistance. Nothing below is defined where this or the effective
    # stress is non-positive, which is the case at and just below the surface.
    qnet = data['qt'] - data['sigma_v']
    valid = (data['sigma_v_eff'] > 0) & (qnet > 0)

    # Iterative calculation of Qtn, Fr, Ic (Eqs. 2-5)
    data['Ic'] = np.where(valid, 2.5, np.nan)  # Initial guess

    for _ in range(10):  # Iterate until convergence
        # Equation 5
        data['n'] = 0.381 * data['Ic'] + 0.05 * (data['sigma_v_eff'] / pa) - 0.15
        data['n'] = data['n'].clip(0.5, 1.0)

        # Equation 2. Robertson's chart runs from Qtn = 1 to Qtn = 1000; a point
        # outside that is masked, not pinned to the edge.
        data['Qtn'] = (qnet / pa) * ((pa / data['sigma_v_eff']) ** data['n'])
        data['Qtn'] = data['Qtn'].where(data['Qtn'].between(1.0, 1000.0))

        # Equation 3. Same treatment for Fr, which the chart runs from 0.1% to 10%.
        data['Fr'] = (data['fs_kpa'] / qnet) * 100.0
        data['Fr'] = data['Fr'].where(data['Fr'].between(0.1, 10.0))

        # Equation 4
        data['Ic'] = np.sqrt((3.47 - np.log10(data['Qtn']))**2 +
                            (np.log10(data['Fr']) + 1.22)**2)

    # Calculate qc1Ncs (Eqs. 6-8)
    data['qc1Ncs'] = np.where(valid, data['qt'] / pa, np.nan)  # Initial estimate

    for _ in range(5):
        # Equation 8. Boulanger and Idriss (2014) give m = 0.782 at qc1Ncs = 21
        # and m = 0.264 at qc1Ncs = 254, so the exponent lives in [0.264, 0.782].
        data['ns'] = 1.338 - 0.249 * (data['qc1Ncs'] ** 0.264)
        data['ns'] = data['ns'].clip(0.264, 0.782)

        # Equation 7
        data['CN'] = (pa / data['sigma_v_eff']) ** data['ns']
        data['CN'] = data['CN'].clip(upper=1.7)

        # Equation 6 - fines correction
        fines_corr = (11.9 + data['CN'] * data['qt'] / (pa * 14.6)) * \
                     np.exp(1.63 - 9.7/(fc + 2) - (15.7/(fc + 2))**2)

        data['qc1Ncs'] = data['CN'] * data['qt'] / pa + fines_corr

    return data

print("Functions defined successfully")


## 7. Fetch All CPT Profiles

In [ ]:
# Fetch and process all CPT profiles
cpt_profiles = {}
errors = []

print("Fetching CPT profiles...")
print("="*100)

for idx, row in tests_df.iterrows():
    scpg_id = row['SCPG_ID']
    test_id = row['TEST_ID']
    site_name = row['SITE_NAME']
    test_name = row['TEST_NAME']
    
    try:
        # Fetch measurements
        df_raw = fetch_cpt_measurements(scpg_id, ngl)
        
        if len(df_raw) < 10:  # Skip if too few measurements
            print(f"⚠ Skipping {test_name} - only {len(df_raw)} measurements")
            continue
        
        # Calculate derived parameters
        df_processed = calculate_derived_parameters(df_raw)
        
        # Store results
        cpt_profiles[test_id] = {
            'site_id': row['SITE_ID'],
            'site_name': site_name,
            'test_id': test_id,
            'test_name': test_name,
            'scpg_id': scpg_id,
            'data': df_processed,
            'n_points': len(df_processed),
            'depth_max': df_processed['depth'].max(),
            'qc_mean': df_processed['qc'].mean(),
            'Ic_mean': df_processed['Ic'].mean()
        }
        
        print(f"✓ {site_name:30s} | {test_name:20s} | "
              f"{len(df_processed):4d} pts | "
              f"Depth: {df_processed['depth'].max():5.1f} m")
        
    except Exception as e:
        errors.append({'test_id': test_id, 'test_name': test_name, 'error': str(e)})
        print(f"✗ Error fetching {test_name}: {str(e)}")

print("\n" + "="*100)
print(f"Successfully fetched: {len(cpt_profiles)} CPT profiles")
print(f"Errors: {len(errors)}")
print("="*100)

## 8. Summary Statistics

In [ ]:
# Create summary DataFrame
summary_data = []
for test_id, profile in cpt_profiles.items():
    df = profile['data']
    summary_data.append({
        'test_id': test_id,
        'site_name': profile['site_name'],
        'test_name': profile['test_name'],
        'n_measurements': profile['n_points'],
        'depth_max_m': profile['depth_max'],
        'qc_min_MPa': df['qc'].min(),
        'qc_max_MPa': df['qc'].max(),
        'qc_mean_MPa': df['qc'].mean(),
        'Ic_min': df['Ic'].min(),
        'Ic_max': df['Ic'].max(),
        'Ic_mean': df['Ic'].mean(),
        'qc1Ncs_min': df['qc1Ncs'].min(),
        'qc1Ncs_max': df['qc1Ncs'].max(),
        'qc1Ncs_mean': df['qc1Ncs'].mean()
    })

summary_df = pd.DataFrame(summary_data)

print("\nCPT Profiles Summary:")
print("="*100)
print(summary_df.to_string(index=False))

print("\n" + "="*100)
print(f"Total profiles: {len(summary_df)}")
print(f"Average depth: {summary_df['depth_max_m'].mean():.1f} m (σ = {summary_df['depth_max_m'].std():.1f} m)")
print(f"Depth range: {summary_df['depth_max_m'].min():.1f} - {summary_df['depth_max_m'].max():.1f} m")
print("="*100)

## 9. Visualize Sample CPT Profiles

In [ ]:
# Plot first 3 CPT profiles
n_plots = min(3, len(cpt_profiles))
test_ids_to_plot = list(cpt_profiles.keys())[:n_plots]

fig, axes = plt.subplots(n_plots, 4, figsize=(16, 4*n_plots), squeeze=False)

for idx, test_id in enumerate(test_ids_to_plot):
    profile = cpt_profiles[test_id]
    df = profile['data']
    title = f"{profile['site_name']} - {profile['test_name']}"
    
    # qc
    axes[idx, 0].plot(df['qc'], df['depth'], 'b-', linewidth=1)
    axes[idx, 0].set_xlabel('qc (MPa)', fontweight='bold')
    axes[idx, 0].set_ylabel('Depth (m)', fontweight='bold')
    axes[idx, 0].invert_yaxis()
    axes[idx, 0].grid(True, alpha=0.3)
    axes[idx, 0].set_title(f'{title}\nCone Tip Resistance', fontsize=10, fontweight='bold')
    
    # Ic
    axes[idx, 1].plot(df['Ic'], df['depth'], 'r-', linewidth=1)
    axes[idx, 1].set_xlabel('Ic', fontweight='bold')
    axes[idx, 1].invert_yaxis()
    axes[idx, 1].grid(True, alpha=0.3)
    axes[idx, 1].axvline(x=2.6, color='gray', linestyle='--', alpha=0.5, label='Ic=2.6')
    axes[idx, 1].set_title('Soil Behavior Type Index', fontsize=10, fontweight='bold')
    axes[idx, 1].legend(fontsize=8)
    
    # qc1Ncs  
    axes[idx, 2].plot(df['qc1Ncs'], df['depth'], 'g-', linewidth=1)
    axes[idx, 2].set_xlabel('qc1Ncs', fontweight='bold')
    axes[idx, 2].invert_yaxis()
    axes[idx, 2].grid(True, alpha=0.3)
    axes[idx, 2].set_title('Normalized Tip Resistance', fontsize=10, fontweight='bold')
    
    # Qtn vs Fr (Robertson chart)
    sc = axes[idx, 3].scatter(df['Fr'], df['Qtn'], c=df['Ic'], 
                              cmap='viridis', s=5, alpha=0.6)
    axes[idx, 3].set_xlabel('Fr (%)', fontweight='bold')
    axes[idx, 3].set_ylabel('Qtn', fontweight='bold')
    axes[idx, 3].set_xscale('log')
    axes[idx, 3].set_yscale('log')
    axes[idx, 3].set_xlim(0.1, 10)
    axes[idx, 3].set_ylim(1, 1000)
    axes[idx, 3].grid(True, alpha=0.3)
    axes[idx, 3].set_title('Qtn vs Fr, coloured by Ic', fontsize=10, fontweight='bold')
    plt.colorbar(sc, ax=axes[idx, 3], label='Ic')

plt.tight_layout()
plt.show()

## 10. Save Data Locally

In [ ]:
# Create data directory
data_dir = 'cpt_data'
os.makedirs(data_dir, exist_ok=True)

# Save each profile
for test_id, profile in cpt_profiles.items():
    # Create safe filename
    site_name = profile['site_name'].replace(' ', '_').replace('/', '_')
    test_name = profile['test_name'].replace(' ', '_').replace('/', '_')
    filename_base = f"{site_name}_{test_name}_T{test_id}"

    # Save as CSV
    csv_path = os.path.join(data_dir, f"{filename_base}.csv")
    profile['data'].to_csv(csv_path, index=False)

    # Save complete profile as pickle
    pkl_path = os.path.join(data_dir, f"{filename_base}.pkl")
    with open(pkl_path, 'wb') as f:
        pickle.dump(profile, f)

# Save the summary and the combined pickle twice: once inside cpt_data/ with the
# per-profile files, and once in the current directory, which is where notebook 06b
# reads them from.
for out_dir in (data_dir, '.'):
    summary_df.to_csv(os.path.join(out_dir, 'cpt_summary.csv'), index=False)
    with open(os.path.join(out_dir, 'all_cpt_profiles.pkl'), 'wb') as f:
        pickle.dump(cpt_profiles, f)

print("\n" + "="*100)
print(f"Saved {len(cpt_profiles)} CPT profiles to '{data_dir}/' directory")
print(f"  - Individual CSV files: {len(cpt_profiles)} files")
print(f"  - Individual pickle files: {len(cpt_profiles)} files")
print(f"  - Summary CSV: cpt_summary.csv (also written to ./ for notebook 06b)")
print(f"  - All profiles pickle: all_cpt_profiles.pkl (also written to ./ for notebook 06b)")
print("="*100)


## 11. Data Ready for Clustering!

The CPT data has been successfully fetched and processed. The data includes:

### Measured Parameters:
- `depth`: Depth (m)
- `qc`: Cone tip resistance (MPa)
- `fs`: Sleeve friction (MPa)
- `u2`: Pore pressure (MPa)

### Derived Parameters (following Hudson et al. 2023):
- `qt`: Corrected tip resistance (kPa)
- `Qtn`: Normalized cone resistance
- `Fr`: Friction ratio (%)
- `Ic`: Soil behavior type index
- `qc1Ncs`: Overburden and fines-corrected tip resistance

### Clustering Features (used in paper):
- **qc1Ncs**: Primary clustering feature
- **Ic**: Secondary clustering feature

These will be standardized before clustering:
```python
q_hat = (qc1Ncs - μ_q) / σ_q
Ic_hat = (Ic - μ_Ic) / σ_Ic
```

### Next Steps:

**Stage 2**: Traditional K-means clustering
- Demonstrate non-contiguous layer problem

**Stage 3**: Agglomerative clustering  
- Implement paper's approach with cost functions
- Compare elbow vs min(J) methods

**Stage 4**: Comparison and evaluation